In [137]:
#  Config 
#  COEQWAL — Water Turbidity Pipeline
#  Algorithm  : Dogliotti et al. (2015) switching model
#  Products   : ACOLITE Aquatic Reflectance C4
#               (Landsat 8/9 + Sentinel-2 A/B/C)
#  Output     : One GeoTiff per target date  [FTU, float32]
#
#  Mirrors the structure of the LST pipeline (COEQWAL_FINAL.py).
#  Run:  python turbidity_pipeline.py
#        python turbidity_pipeline.py --config other_config.yaml



# ── IMPORTS ──────────────────────────────────────────────────────────────────
import ast
import logging
import os
import re
import sys
import zipfile
from pathlib import Path

import datacube
import geopandas as gpd
import numpy as np
import pandas as pd
import rioxarray                        # noqa: F401  (activates .rio accessor)
import xarray as xr
import yaml
from rasterio.env import Env
from rioxarray.merge import merge_arrays


# ── LOGGING ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("coeqwal_turb")


# ── CONFIG ───────────────────────────────────────────────────────────────────
_DEFAULT_CONFIG = "config_turbidity.yaml"

def load_config(path: str = _DEFAULT_CONFIG) -> dict:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Config not found: {p.resolve()}")
    with open(p) as f:
        return yaml.safe_load(f)

_cfg_path = _DEFAULT_CONFIG
if "--config" in sys.argv:
    idx = sys.argv.index("--config")
    if idx + 1 < len(sys.argv):
        _cfg_path = sys.argv[idx + 1]

CFG = load_config(_cfg_path)

# Paths
INPUTS_DIR  = Path("inputs")
OUTPUTS_DIR = Path("outputs")

AOI_FILE    = Path(CFG["aoi_file"])
TARGETS_CSV = Path(CFG["targets_csv"])

OUT_DIR_TIFS     = Path(CFG["outputs"]["tifs_dir"])
OUT_COVERAGE_CSV = Path(CFG["outputs"]["coverage_csv"])
OUT_EFFECTIVE_CSV= Path(CFG["outputs"]["effective_csv"])
OUT_SUMMARY      = Path(CFG["outputs"]["run_summary_csv"])
OUT_ZIP          = Path(CFG["outputs"]["zip_file"])

# Spatial
OUTPUT_CRS          = CFG["output_crs"]
SENTINEL_RESOLUTION = tuple(CFG["sentinel_resolution"])
LANDSAT_RESOLUTION  = tuple(CFG["landsat_resolution"])
MGRS_TILES          = set(CFG["mgrs_tiles"])
WRS_PATH            = CFG["wrs_path"]
WRS_ROWS            = set(CFG["wrs_rows"])

# Search
SEARCH_WINDOW_DAYS = CFG["search_window_days"]
CLOUD_COVER_MAX    = CFG.get("cloud_cover_max")
SENSOR_FAMILIES    = CFG["sensor_families"]
# Flat lookup: product_name substring → family key
_FAMILY_OF = {}
for _fam, _fcfg in SENSOR_FAMILIES.items():
    for _key in _fcfg["preference"]:
        _FAMILY_OF[_key] = _fam

# Products & bands
PRODUCTS = CFG["products"]
BAND_MAP = CFG["band_map"]          # dict[product_name -> {red, nir, swir1, flags}]

# Scaling
_sc            = CFG["scaling"]
SCALE          = float(_sc["scale"])
OFFSET         = float(_sc["offset"])
NODATA_RAW     = int(_sc["nodata_raw"])     # 65535
FLAGS_NODATA   = int(_sc["flags_nodata"])   # 16

# Masking
LAND_SWIR_THRESHOLD = float(CFG["land_swir_threshold"])
FLAGS_INVALID_BITS  = int(CFG["flags_invalid_bits"])   # 24 = 8|16
RRS_MIN             = float(CFG["rrs_min"])
RRS_MAX             = float(CFG["rrs_max"])
MIN_VALID_PX_TILE   = int(CFG.get("min_valid_pixels_per_tile", 0))

# Dogliotti
_dog         = CFG["dogliotti"]
DOG_A_RED    = float(_dog["a_red"])
DOG_C_RED    = float(_dog["c_red"])
DOG_A_NIR    = float(_dog["a_nir"])
DOG_C_NIR    = float(_dog["c_nir"])
SWITCH_LOW   = float(_dog["switch_low"])
SWITCH_HIGH  = float(_dog["switch_high"])
TURB_MAX_FTU = float(_dog["turb_max_ftu"])

# Output raster
NODATA_OUT = float(CFG["nodata_out"])
COMPRESS   = CFG.get("compress", "LZW")

# AWS
_aws = CFG.get("aws", {})
os.environ["AWS_REQUEST_PAYER"]  = _aws.get("request_payer", "requester")
os.environ["AWS_DEFAULT_REGION"] = _aws.get("default_region", "us-west-2")
os.environ["AWS_REGION"]         = _aws.get("default_region", "us-west-2")
os.environ.pop("AWS_NO_SIGN_REQUEST", None)
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"]     = "YES"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif,.TIF,.xml,.XML"
os.environ["VSI_CACHE"]      = "TRUE"
os.environ["VSI_CACHE_SIZE"] = str(128 * 1024 * 1024)

logger.info("AWS_REQUEST_PAYER  : %s", os.environ["AWS_REQUEST_PAYER"])
logger.info("AWS_DEFAULT_REGION : %s", os.environ["AWS_DEFAULT_REGION"])


2026-05-06 01:11:55  INFO      AWS_REQUEST_PAYER  : requester
2026-05-06 01:11:55  INFO      AWS_DEFAULT_REGION : us-west-2


In [138]:
#  CHUNK 0 — Shared utilities
#  (same logic as LST pipeline; kept self-contained here)

# ── AOI helpers ──────────────────────────────────────────────
_AOI_CACHE_4326 = None

def load_aoi_4326(aoi_path: Path = AOI_FILE) -> gpd.GeoDataFrame:
    global _AOI_CACHE_4326
    if _AOI_CACHE_4326 is not None:
        return _AOI_CACHE_4326
    if not aoi_path.exists():
        raise FileNotFoundError(f"AOI not found: {aoi_path.resolve()}")
    aoi = gpd.read_file(aoi_path)
    if aoi.crs is None:
        raise ValueError("AOI has no CRS.")
    if aoi.crs.to_string() != "EPSG:4326":
        aoi = aoi.to_crs("EPSG:4326")
    if (~aoi.is_valid).any():
        aoi["geometry"] = aoi.geometry.buffer(0)
        aoi = aoi[aoi.geometry.notnull()].copy()
    if len(aoi) > 1:
        aoi = aoi.dissolve().reset_index(drop=True)
    _AOI_CACHE_4326 = aoi
    return aoi

def get_bbox_wgs84() -> dict:
    aoi = load_aoi_4326()
    minx, miny, maxx, maxy = aoi.total_bounds
    return {"x": (minx, maxx), "y": (miny, maxy)}

def get_aoi_in_crs(dst_crs: str = OUTPUT_CRS) -> gpd.GeoDataFrame:
    return load_aoi_4326().to_crs(dst_crs)


# ── Time helpers ─────────────────────────────────────────────
def _date_range(center_date, days):
    c = pd.Timestamp(center_date).normalize()
    return (c - pd.Timedelta(days=days), c + pd.Timedelta(days=days + 1))

def _as_utc_naive(ts):
    ts = pd.Timestamp(ts)
    if ts.tz is not None:
        return ts.tz_convert("UTC").tz_localize(None)
    return ts

def _extract_scene_datetime(ds):
    md = getattr(ds, "metadata_doc", None) or {}
    if isinstance(md, dict):
        dt = (md.get("properties", {}) or {}).get("datetime", None)
        if dt:
            try:
                return _as_utc_naive(pd.to_datetime(dt))
            except Exception as e:
                logger.debug("Cannot parse datetime %r: %s", dt, e)
    ct = getattr(ds, "center_time", None)
    if ct is not None:
        try:
            return _as_utc_naive(pd.to_datetime(ct))
        except Exception as e:
            logger.debug("Cannot parse center_time %r: %s", ct, e)
    try:
        return _as_utc_naive(pd.to_datetime(ds.metadata.time))
    except Exception:
        return None

def _extract_cloud_cover(ds) -> float | None:
    md = getattr(ds, "metadata_doc", None) or {}
    if isinstance(md, dict):
        cc = (md.get("properties", {}) or {}).get("eo:cloud_cover", None)
        if cc is not None:
            try:
                return float(cc)
            except Exception:
                return None
    return None


# ── Product / sensor helpers ──────────────────────────────────
def _family_of(product_name: str) -> str | None:
    """Return 'landsat' or 'sentinel2' for a given product name."""
    p = product_name.lower()
    for key, fam in _FAMILY_OF.items():
        if key in p:
            return fam
    return None

def _sensor_priority_in_family(product_name: str) -> int:
    """
    Lower = preferred within its family.
    Uses the order defined in sensor_families.<family>.preference.
    """
    fam = _family_of(product_name)
    if fam is None:
        return 999
    p = product_name.lower()
    for i, key in enumerate(SENSOR_FAMILIES[fam]["preference"], start=1):
        if key in p:
            return i
    return 999

def _is_sentinel(product_name: str) -> bool:
    return "s2" in product_name.lower()

def _resolution_for(product_name: str) -> tuple:
    return SENTINEL_RESOLUTION if _is_sentinel(product_name) else LANDSAT_RESOLUTION

def _mgrs_code(ds) -> str | None:
    """Extract the MGRS tile code from dataset metadata (e.g. 'MGRS-10SFH' → '10SFH')."""
    md   = getattr(ds, "metadata_doc", None) or {}
    code = (md.get("properties", {}) or {}).get("odc:region_code", None)
    if code:
        return code.replace("MGRS-", "")
    return None

def _wrs_path_row(ds) -> tuple[int | None, int | None]:
    md    = getattr(ds, "metadata_doc", None) or {}
    props = (md.get("properties", {}) or {})
    for kp, kr in [("landsat:wrs_path", "landsat:wrs_row"), ("wrs_path", "wrs_row")]:
        p, r = props.get(kp), props.get(kr)
        try:
            if p is not None and r is not None:
                return int(p), int(r)
        except Exception:
            pass
    return None, None

def _get_ds_by_uuid(dc: datacube.Datacube, uid: str):
    import uuid as _uuid
    return dc.index.datasets.get(_uuid.UUID(uid))


In [139]:
#  CHUNK 1 — Radiometric scaling

def scale_rrs(raw: np.ndarray) -> np.ndarray:
    """
    Convert raw uint16 integer to Rrs (sr⁻¹).
    1) Replace NODATA_RAW with NaN
    2) Apply:  Rrs = raw * SCALE + OFFSET
    3) Clamp to physical range [RRS_MIN, RRS_MAX]
    """
    arr = raw.astype(np.float32)
    arr = np.where(arr == NODATA_RAW, np.nan, arr)
    arr = arr * SCALE + OFFSET
    arr = np.where((arr < RRS_MIN) | (arr > RRS_MAX), np.nan, arr)
    return arr


In [140]:
#  CHUNK 2 — Masking

def build_water_mask(
    rrs_swir1: np.ndarray,
    flags_raw: np.ndarray,
) -> np.ndarray:
    """
    Returns boolean array: True = valid water pixel.

    Three independent gates (AND logic):
      1) SWIR land test   : Rrs_swir1 <= LAND_SWIR_THRESHOLD
      2) ACOLITE l2_flags : (flags & FLAGS_INVALID_BITS) == 0
      3) Implicit         : NaN propagation from scale_rrs handles
                            nodata and out-of-range reflectances.

    Parameters
    ----------
    rrs_swir1 : scaled Rrs array for SWIR1 band  (float32, may contain NaN)
    flags_raw : raw uint8 flags array             (may contain FLAGS_NODATA)
    """
    # Gate 1: water vs land via SWIR
    water = (rrs_swir1 <= LAND_SWIR_THRESHOLD) & ~np.isnan(rrs_swir1)

    # Gate 2: ACOLITE quality flags
    f = flags_raw.astype(np.uint8)
    valid_flags = (f & np.uint8(FLAGS_INVALID_BITS)) == 0

    return water & valid_flags


In [141]:
#  CHUNK 3 — Dogliotti turbidity algorithm

def _rrs_to_rho(rrs: np.ndarray) -> np.ndarray:
    """Remote sensing reflectance → water reflectance.  rho = pi * Rrs"""
    return np.pi * rrs

def _turb_arm(rho: np.ndarray, A: float, C: float) -> np.ndarray:
    """Single-band Dogliotti arm.  T = A * rho / (1 - rho/C)"""
    denominator = 1.0 - rho / C
    # Denominator ≤ 0 means rho ≥ C: physically invalid → NaN
    return np.where(
        (denominator <= 0) | (rho < 0),
        np.nan,
        A * rho / denominator,
    )

def turbidity_dogliotti(
    rrs_red: np.ndarray,
    rrs_nir: np.ndarray,
) -> np.ndarray:
    """
    Dogliotti et al. (2015) switching turbidity model.

    Low turbidity  (rho_red < SWITCH_LOW)  → red arm only
    High turbidity (rho_red > SWITCH_HIGH) → NIR arm only
    Transition     (between thresholds)    → weighted blend

    Additional safeguard: results > TURB_MAX_FTU are physically
    implausible.  If the red arm alone is < TURB_MAX_FTU, fall back
    to it; otherwise set NaN.

    Returns turbidity in FTU (float32).
    """
    rho_red = _rrs_to_rho(rrs_red)
    rho_nir = _rrs_to_rho(rrs_nir)

    T_red = _turb_arm(rho_red, DOG_A_RED, DOG_C_RED)
    T_nir = _turb_arm(rho_nir, DOG_A_NIR, DOG_C_NIR)

    # Blend weight: 0 at SWITCH_LOW, 1 at SWITCH_HIGH
    w     = np.clip((rho_red - SWITCH_LOW) / (SWITCH_HIGH - SWITCH_LOW), 0.0, 1.0)
    blend = (1.0 - w) * T_red + w * T_nir

    turb = np.where(
        rho_red < SWITCH_LOW,  T_red,
        np.where(rho_red > SWITCH_HIGH, T_nir, blend)
    )

    # Cap implausible values (from colleague's improvement)
    turb = np.where(
        (turb > TURB_MAX_FTU) & (T_red < TURB_MAX_FTU),
        T_red,
        turb,
    )
    turb = np.where(turb > TURB_MAX_FTU, np.nan, turb)

    return turb.astype(np.float32)



In [142]:
#  CHUNK 4 — Single-dataset loader + processor
#
#  Returns a dict with:
#    "turb"     : xr.DataArray  (turbidity FTU, clipped to AOI)
#    "n_valid"  : int
#    "turb_min" : float
#    "turb_max" : float
#    "turb_mean": float

def process_dataset(
    dc: datacube.Datacube,
    ds_odc,                         # ODC Dataset object
    product: str,
    aoi_out: gpd.GeoDataFrame,      # AOI in OUTPUT_CRS
) -> dict:
    """
    Load one ODC dataset, apply full masking pipeline, compute
    turbidity via Dogliotti, clip to AOI.

    Raises ValueError if the result has 0 valid pixels.
    """
    bmap       = BAND_MAP[product]
    resolution = _resolution_for(product)
    bands_needed = [bmap["red"], bmap["nir"], bmap["swir1"], bmap["flags"]]

    with Env(AWS_REQUEST_PAYER="requester", GDAL_DISABLE_READDIR_ON_OPEN="YES"):
        raw = dc.load(
            datasets=[ds_odc],
            measurements=bands_needed,
            output_crs=OUTPUT_CRS,
            resolution=resolution,
            group_by="solar_day",
        )

    if len(raw.time) == 0:
        raise ValueError("dc.load returned empty dataset.")

    # Take first time slice (should only be one after group_by="solar_day")
    raw = raw.isel(time=0)

    # ── Scale Rrs bands ───────────────────────────────────────
    rrs_red  = scale_rrs(raw[bmap["red"]].values)
    rrs_nir  = scale_rrs(raw[bmap["nir"]].values)
    rrs_swir = scale_rrs(raw[bmap["swir1"]].values)
    flags    = raw[bmap["flags"]].values

    # ── Build water mask ──────────────────────────────────────
    mask = build_water_mask(rrs_swir, flags)

    # ── Dogliotti turbidity ───────────────────────────────────
    # Apply mask first so invalid pixels don't enter the algorithm
    rrs_red_m = np.where(mask, rrs_red, np.nan)
    rrs_nir_m = np.where(mask, rrs_nir, np.nan)

    turb_arr = turbidity_dogliotti(rrs_red_m, rrs_nir_m)

    # ── Wrap in DataArray inheriting spatial coords ───────────
    ref   = raw[bmap["red"]]
    turb_da = xr.DataArray(
        turb_arr,
        dims=["y", "x"],
        coords={"y": ref.coords["y"], "x": ref.coords["x"]},
    )
    turb_da = turb_da.rio.write_crs(OUTPUT_CRS)

    # ── Clip to AOI ───────────────────────────────────────────
    turb_clip = turb_da.rio.clip(aoi_out.geometry, aoi_out.crs, drop=True)

    n_valid  = int(np.sum(np.isfinite(turb_clip.values)))
    if n_valid == 0:
        raise ValueError("0 valid pixels after AOI clip.")

    finite   = turb_clip.values[np.isfinite(turb_clip.values)]
    return {
        "turb":      turb_clip,
        "n_valid":   n_valid,
        "turb_min":  float(np.nanmin(finite)),
        "turb_max":  float(np.nanmax(finite)),
        "turb_mean": float(np.nanmean(finite)),
    }



In [143]:
#  CHUNK 5 — Scene discovery (Sentinel-2 and Landsat)

def discover_scenes(dc: datacube.Datacube) -> pd.DataFrame:
    """
    For every target date, search all products within
    ±SEARCH_WINDOW_DAYS.  Returns a DataFrame of candidates.
    """
    sel = pd.read_csv(TARGETS_CSV, parse_dates=["Date"])
    target_dates = sorted({pd.Timestamp(d).date() for d in sel["Date"].dropna()})
    logger.info("Loaded %d target dates from %s", len(target_dates), TARGETS_CSV)

    bbox = get_bbox_wgs84()
    rows = []

    # Verify which products exist in this ODC
    available = set(dc.list_products().index.astype(str))
    valid_products = [p for p in PRODUCTS if p in available]
    logger.info("Valid products: %s", valid_products)

    for td in target_dates:
        t0, t1 = _date_range(td, SEARCH_WINDOW_DAYS)
        for prod in valid_products:
            dss = dc.find_datasets(product=prod, time=(t0, t1), **bbox)
            for ds in dss:
                # ── Tile filter ──────────────────────────────
                if _is_sentinel(prod):
                    mgrs = _mgrs_code(ds)
                    if mgrs not in MGRS_TILES:
                        continue
                else:
                    path, row = _wrs_path_row(ds)
                    if path != WRS_PATH or row not in WRS_ROWS:
                        continue

                # ── Cloud cover filter ───────────────────────
                cc = _extract_cloud_cover(ds)
                if CLOUD_COVER_MAX is not None and cc is not None:
                    if cc > CLOUD_COVER_MAX:
                        continue

                dt = _extract_scene_datetime(ds)
                if dt is None:
                    continue
                scene_day = _as_utc_naive(dt).normalize()
                offset    = int((scene_day - pd.Timestamp(td)).days)

                rows.append({
                    "TargetDate":   td.isoformat(),
                    "Product":      prod,
                    "SceneDate":    scene_day.date().isoformat(),
                    "OffsetDays":   offset,
                    "CloudCover":   cc,
                    "TileCode":     _mgrs_code(ds) if _is_sentinel(prod) else
                                    str(_wrs_path_row(ds)),
                    "SensorPriority": _sensor_priority_in_family(prod),
                    "SensorFamily": _family_of(prod),
                    "ODC_id":       str(ds.id),
                })

    candidates = pd.DataFrame(rows)
    logger.info("Total candidates found: %d", len(candidates))
    OUTPUTS_DIR.mkdir(exist_ok=True)
    candidates.to_csv(OUT_COVERAGE_CSV, index=False)
    logger.info("Coverage CSV: %s", OUT_COVERAGE_CSV)
    return candidates



In [144]:
#  CHUNK 6 — Best-scene selector per target date, per family
#
#  Strategy:
#    For each TargetDate × SensorFamily:
#      1) Filter candidates to that family
#      2) For each (Product, SceneDate) combo check tile completeness:
#           Landsat  → both WRS rows 33 + 34 present
#           Sentinel → all 4 MGRS tiles present
#      3) Score complete scenes by (|OffsetDays|, SensorPriority, CloudCover)
#      4) Keep the winner
#
#    Result: up to 2 rows per TargetDate (one Landsat + one Sentinel-2).
#    If a family has no complete scene → that family is skipped silently.

def select_best_scenes(candidates: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a DataFrame with up to 2 rows per TargetDate
    (one per sensor family that has a complete, valid scene).
    """
    if candidates.empty:
        logger.warning("No candidates — nothing to select.")
        return pd.DataFrame()

    records = []

    for td, td_grp in candidates.groupby("TargetDate"):

        for fam, fam_cfg in SENSOR_FAMILIES.items():

            fam_grp = td_grp[td_grp["SensorFamily"] == fam]
            if fam_grp.empty:
                logger.debug("[%s][%s] No candidates.", td, fam)
                continue

            best = None

            for (prod, scene_date), sg in fam_grp.groupby(["Product", "SceneDate"]):

                # ── Completeness check ────────────────────────
                if fam == "landsat":
                    # Group tiles by path and check each path has both rows
                    path_rows = {}
                    for tc in sg["TileCode"]:
                        m = re.search(r"(\d+),\s*(\d+)", str(tc))
                        if m:
                            p, r = int(m.group(1)), int(m.group(2))
                            path_rows.setdefault(p, set()).add(r)

                    # Find paths that have both rows complete
                    complete_paths = [
                        p for p, rows in path_rows.items()
                        if WRS_ROWS.issubset(rows)
                    ]
                    if not complete_paths:
                        logger.debug(
                            "[%s][%s] %s %s: no path has complete rows %s — skip",
                            td, fam, prod, scene_date, path_rows,
                        )
                        continue

                    # Keep only tiles from the first complete path
                    best_path = complete_paths[0]
                    sg = sg[sg["TileCode"].apply(
                        lambda tc: bool(re.search(rf"\({best_path},", str(tc)))
                    )]

                # ── Score ─────────────────────────────────────
                offset   = int(sg["OffsetDays"].iloc[0])
                sen_prio = int(sg["SensorPriority"].iloc[0])
                cc       = float(sg["CloudCover"].mean()) if sg["CloudCover"].notna().any() else 100.0
                odc_ids  = sg["ODC_id"].tolist()

                score     = (abs(offset), sen_prio, cc)
                candidate = {
                    "TargetDate":     td,
                    "SensorFamily":   fam,
                    "Product":        prod,
                    "SceneDate":      scene_date,
                    "OffsetDays":     offset,
                    "AbsOffset":      abs(offset),
                    "SensorPriority": sen_prio,
                    "CloudCover":     cc,
                    "ODC_ids":        odc_ids,
                    "_score":         score,
                }

                if best is None or score < best["_score"]:
                    best = candidate

            if best is None:
                logger.info(
                    "[%s][%s] No complete scene found — this family skipped.",
                    td, fam,
                )
                continue

            del best["_score"]
            records.append(best)
            logger.info(
                "[%s][%s] Best → %s  SceneDate=%s  Offset=%+d  Cloud=%.1f%%",
                td, fam, best["Product"], best["SceneDate"],
                best["OffsetDays"], best["CloudCover"],
            )

    effective = pd.DataFrame(records)

    if not effective.empty:
        n_dates     = effective["TargetDate"].nunique()
        both        = effective.groupby("TargetDate")["SensorFamily"].nunique()
        n_both      = int((both == 2).sum())
        n_one       = int((both == 1).sum())
        logger.info(
            "Effective scenes: %d rows for %d dates  "
            "(%d dates with both families, %d with one only)",
            len(effective), n_dates, n_both, n_one,
        )

    effective.to_csv(OUT_EFFECTIVE_CSV, index=False)
    logger.info("Effective CSV: %s", OUT_EFFECTIVE_CSV)
    return effective


In [145]:
#  CHUNK 7 — Mosaicker
#
#  For each effective scene:
#    • Load each individual ODC dataset (tile) separately
#    • Process each tile with process_dataset()
#    • Merge all tiles with merge_arrays() (first-valid-pixel)
#    • Write final TIF
 
def mosaic_and_write(
    dc: datacube.Datacube,
    effective: pd.DataFrame,
) -> list[dict]:
    """
    Main processing loop.  Returns list of run_row dicts for the
    summary CSV (mirrors the LST pipeline run_rows pattern).
    """
    OUT_DIR_TIFS.mkdir(parents=True, exist_ok=True)
    aoi_out = get_aoi_in_crs(OUTPUT_CRS)
    run_rows = []
 
    for _, row in effective.iterrows():
        
        td          = row["TargetDate"]
        prod        = row["Product"]
        scene_date  = row["SceneDate"]
        offset_days = row["OffsetDays"]
        fam         = row.get("SensorFamily", "unknown")

        # ODC_ids puede ser lista Python (DataFrame en memoria)
        # o string tipo "['uuid1', 'uuid2']" (leído desde CSV).
        raw_ids = row["ODC_ids"]
        if isinstance(raw_ids, list):
            odc_ids = raw_ids
        else:
            try:
                odc_ids = ast.literal_eval(str(raw_ids))
            except Exception:
                odc_ids = [x.strip() for x in str(raw_ids).split("|") if x.strip()]
 
        tile_arrays  = []
        tile_stats   = []
        tiles_rejected = 0
 
        for uid in odc_ids:
            ds_odc = _get_ds_by_uuid(dc, uid)
            if ds_odc is None:
                logger.warning("  Dataset %s not found in ODC index — skip tile.", uid)
                tiles_rejected += 1
                continue
            try:
                result = process_dataset(dc, ds_odc, prod, aoi_out)
 
                # ── Minimum valid pixels per tile ─────────────
                if MIN_VALID_PX_TILE > 0 and result["n_valid"] < MIN_VALID_PX_TILE:
                    logger.warning(
                        "  Tile %s  valid_px=%d < threshold=%d "
                        "(likely cloudy) — tile rejected.",
                        uid[:8], result["n_valid"], MIN_VALID_PX_TILE,
                    )
                    tiles_rejected += 1
                    tile_stats.append({
                        "ODC_id":  uid,
                        "n_valid": result["n_valid"],
                        "min":     result["turb_min"],
                        "max":     result["turb_max"],
                        "mean":    result["turb_mean"],
                        "rejected": True,
                    })
                    continue
 
                tile_arrays.append(result["turb"])
                tile_stats.append({
                    "ODC_id":   uid,
                    "n_valid":  result["n_valid"],
                    "min":      result["turb_min"],
                    "max":      result["turb_max"],
                    "mean":     result["turb_mean"],
                    "rejected": False,
                })
                logger.info("  Tile %s  valid_px=%d  FTU min/max/mean=%.1f/%.1f/%.1f",
                            uid[:8], result["n_valid"],
                            result["turb_min"], result["turb_max"], result["turb_mean"])
            except Exception as e:
                logger.warning("  Tile %s failed: %s", uid[:8], e)
                tiles_rejected += 1
 
        if not tile_arrays:
            logger.warning("  SKIP %s [%s]: all tiles failed.", td, fam)
            run_rows.append(_run_row(td, prod, scene_date, offset_days,
                                     odc_ids, "", "skipped_all_tiles_failed", fam=fam))
            continue
 
        # ── Merge tiles into mosaic (first-valid-pixel) ───────
        if len(tile_arrays) == 1:
            mosaic = tile_arrays[0]
        else:
            mosaic = merge_arrays(tile_arrays, nodata=np.nan, method="first")
 
        mosaic = mosaic.rio.write_crs(OUTPUT_CRS)
        n_mosaic = int(np.sum(np.isfinite(mosaic.values)))
 
        if n_mosaic == 0:
            logger.warning("  SKIP %s [%s]: 0 valid pixels in merged mosaic.", td, fam)
            run_rows.append(_run_row(td, prod, scene_date, offset_days,
                                     odc_ids, "", "skipped_empty_mosaic", fam=fam))
            continue
 
        # ── Determine coverage completeness ───────────────────
        n_expected = len(odc_ids)
        n_accepted = len(tile_arrays)
        coverage_complete = (tiles_rejected == 0)
 
        if not coverage_complete:
            logger.warning(
                "  Incomplete coverage: %d/%d tiles accepted "
                "(%d rejected by pixel threshold or error).",
                n_accepted, n_expected, tiles_rejected,
            )
 
        finite_vals = mosaic.values[np.isfinite(mosaic.values)]
        logger.info("  Mosaic  valid_px=%d  FTU min/max/mean=%.1f/%.1f/%.1f",
                    n_mosaic, float(np.nanmin(finite_vals)),
                    float(np.nanmax(finite_vals)), float(np.nanmean(finite_vals)))
 
        # ── Write TIF ─────────────────────────────────────────
        fam_tag = row.get("SensorFamily", "unknown")
        out_tif = OUT_DIR_TIFS / (
            f"turb_delta_{td}_scene_{scene_date}_{prod}_{fam_tag}.tif"
        )
        out_arr = mosaic.astype("float32")
        out_arr = out_arr.where(np.isfinite(out_arr), NODATA_OUT)
        out_arr.rio.write_nodata(NODATA_OUT, inplace=True)
        out_arr.rio.to_raster(out_tif, nodata=NODATA_OUT, compress=COMPRESS)
        logger.info("  Wrote: %s", out_tif)
 
        tif_status = "written" if coverage_complete else "written_incomplete"
 
        run_rows.append(_run_row(
            td, prod, scene_date, offset_days, odc_ids,
            str(out_tif), tif_status,
            n_mosaic=n_mosaic,
            turb_min=float(np.nanmin(finite_vals)),
            turb_max=float(np.nanmax(finite_vals)),
            turb_mean=float(np.nanmean(finite_vals)),
            tile_stats=tile_stats,
            fam=fam,
        ))
 
    return run_rows
 
 
def _run_row(td, prod, scene_date, offset_days, odc_ids, out_tif, status,
             n_mosaic=None, turb_min=None, turb_max=None, turb_mean=None,
             tile_stats=None, fam=None):
    """Build a summary row dict (mirrors LST pipeline pattern)."""
    return {
        "TargetDate":   td,
        "SensorFamily": fam,
        "SceneDate":    scene_date,
        "Product":      prod,
        "OffsetDays":   offset_days,
        "ODC_ids":      "|".join(odc_ids),
        "n_tiles":      len(odc_ids),
        "OutTIF":       out_tif,
        "status":       status,
        "mosaic_valid_px":  n_mosaic,
        "mosaic_turb_min":  turb_min,
        "mosaic_turb_max":  turb_max,
        "mosaic_turb_mean": turb_mean,
        "tile_stats":   str(tile_stats) if tile_stats else "",
        "OUTPUT_CRS":   OUTPUT_CRS,
        "RESOLUTION":   str(SENTINEL_RESOLUTION if _is_sentinel(prod) else LANDSAT_RESOLUTION),
        "LAND_SWIR_THRESHOLD": LAND_SWIR_THRESHOLD,
        "FLAGS_INVALID_BITS":  FLAGS_INVALID_BITS,
    }


In [146]:
#  CHUNK 8 — Summary + ZIP  (mirrors LST CHUNK 8E)

def write_summary_and_zip(run_rows: list[dict]):
    summary = pd.DataFrame(run_rows)
    summary.to_csv(OUT_SUMMARY, index=False)
    logger.info("Run summary: %s  (rows=%d)", OUT_SUMMARY, len(summary))

    written = summary.loc[summary["status"] == "written", "OutTIF"].tolist()

    with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for p in written:
            if p and Path(p).exists():
                z.write(p, arcname=Path(p).name)

    logger.info("Written TIFs: %d", len(written))
    logger.info("ZIP: %s", OUT_ZIP)




In [147]:
#  MAIN

def main():
    logger.info("=" * 70)
    logger.info("COEQWAL — Turbidity Pipeline  |  Dogliotti ACOLITE AR")
    logger.info("Config: %s", _cfg_path)
    logger.info("=" * 70)

    dc = datacube.Datacube()

    # 1) Discover all candidate scenes
    candidates = discover_scenes(dc)

    if candidates.empty:
        logger.error("No candidates found. Check products, dates, and bbox.")
        sys.exit(1)

    # 2) Select best scene per target date
    effective = select_best_scenes(candidates)

    if effective.empty:
        logger.error("No complete scenes selected. "
                     "Check tile coverage and cloud cover thresholds.")
        sys.exit(1)

    # 3) Process each scene, mosaic tiles, write TIFs
    run_rows = mosaic_and_write(dc, effective)

    # 4) Write summary CSV + ZIP
    write_summary_and_zip(run_rows)

    logger.info("=" * 70)
    logger.info("DONE")
    logger.info("=" * 70)


if __name__ == "__main__":
    main()


2026-05-06 01:11:57  INFO      ======================================================================
2026-05-06 01:11:57  INFO      COEQWAL — Turbidity Pipeline  |  Dogliotti ACOLITE AR
2026-05-06 01:11:57  INFO      Config: config_turbidity.yaml
2026-05-06 01:11:57  INFO      ======================================================================
2026-05-06 01:11:57  INFO      Loaded 3 target dates from inputs/target_dates.csv
2026-05-06 01:11:57  INFO      Valid products: ['landsat8_c2_acolite_ar_c4', 'landsat9_c2_acolite_ar_c4', 's2a_acolite_ar_c4', 's2b_acolite_ar_c4', 's2c_acolite_ar_c4']
2026-05-06 01:12:18  INFO      Total candidates found: 54
2026-05-06 01:12:18  INFO      Coverage CSV: outputs/satellite_coverage_report_turbidity.csv
2026-05-06 01:12:18  INFO      [2017-08-11][landsat] No complete scene found — this family skipped.
2026-05-06 01:12:18  INFO      [2017-08-11][sentinel2] Best → s2b_acolite_ar_c4  SceneDate=2017-08-13  Offset=+2  Cloud=0.5%
2026-05-06 01:12:18  IN